# 05 — job header

Explore capture **3521**: the `/flagship-web/jobs/search-results/` response that embeds the job summary card (title, company, location, badges, apply button).

Target extractor: `adapters/linkedin/extract/job_header.py`.

Run from repo root.

In [1]:
import json
import re
from urllib.parse import parse_qs, urlparse

from sqlalchemy import text

from adapters.linkedin import rsc
from core.db import SessionLocal

CAPTURE_ID = 3521

query = text("""
    SELECT id, request_url, request_body, response_body
    FROM mitm_http_captures
    WHERE id = :capture_id
""")

with SessionLocal() as session:
    row = session.execute(query, {"capture_id": CAPTURE_ID}).fetchone()

if row is None:
    raise SystemExit(f"capture {CAPTURE_ID} not found")

capture_id, request_url, request_body, response_body = row
print("capture_id:", capture_id)
print("path:", urlparse(request_url).path)
print("response chars:", len(response_body))

capture_id: 3521
path: /flagship-web/jobs/search-results/
response chars: 381043


## request URL

The job id is in the query string as `currentJobId`, not in the SDUI component request body.

In [2]:
print(request_url)

job_id = parse_qs(urlparse(request_url).query)["currentJobId"][0]
print("currentJobId:", job_id)

https://www.linkedin.com/flagship-web/jobs/search-results/?currentJobId=4430365784&eBP=CwEAAAGfEPg4OE3hhlQE8rYiz7VVBtRZhhdL2f-bGcUYIF8MzRHeJidG2eNG6HP4ztP3i38Vltllw_M-6uKlt4Uk9kufCJ6ESfi2DwB07cad6yQolWPRbGeHOrobhrmSM6Ai1rZwvK5AQ19hEwi8PXg8RRNHgt4jAkdpId39vUzYQahGodPY5zvmkiTl-r-pDDj4VYBUYeR9p-HRe7m_4JeaInvaZcplRtjf1lhnDxq03X4NsOBoBkVqlfpCTEoGYTJ_PC6OdbzMzFobcNdK-Gt1_DKelo-cH7mdvuihlZTdYnh99nfb28NT1Y5MIlwux12FRwyY3MnHhLyLr9i_1dopggqT0kBCFuofD4C1dOiOV7SFaPRBriQYUZAw9iviSjVkcf8-d5rbhC1cG6wGAtmeITx-b5Ar015MhjQBzJ01C4sw9z-Qz3OASGOfOO8vm6-JHPpVfJPtpnFgDVziA-IzGgjt&refId=72Kf2j7nEovdsPVhxJFRzw%3D%3D&trackingId=gomPAqwiJQfhZqZqb8oFUw%3D%3D&keywords=forward+deployed+engineer&origin=SEMANTIC_SEARCH_LANDING_PAGE
currentJobId: 4430365784


## request body

Unlike component fetches (see notebook 02), this is a screen navigation payload — not `clientArguments.payload.jobId`.

In [3]:
req = json.loads(request_body)
print("top-level keys:", sorted(req.keys()))
print("screenId:", req.get("screenId"))
print("pageKey:", req.get("pageKey"))

top-level keys: ['$type', 'clearBackStack', 'colorScheme', 'disableScreenGutters', 'inheritActor', 'newHierarchy', 'pageKey', 'presentation', 'presentationStyle', 'replaceCurrentScreen', 'requestedArguments', 'screenId', 'screenTitle', 'shouldHideLoadingSpinner', 'shouldHideMobileTopNavBar', 'shouldHideMobileTopNavBarDivider', 'title', 'url']
screenId: com.linkedin.sdui.flagshipnav.jobs.SemanticJobDetails
pageKey: nlsearch_srp_jobs


## RSC stream overview

Same wire format as notebook 03: multiline `<chunk_id>:<json>`.

In [4]:
chunks = rsc.parse_stream(response_body)
print(len(chunks), "chunks")
print("first ids:", list(chunks.keys())[:15])

285 chunks
first ids: ['1', '3', '4', '5', '7', '8', '9', 'b', 'c', '2d', '2f', '31', '32', '33', '34']


## locate header strings

Capture 3521 is for **Forward Deployed Engineer** at **appliers.ai** (job `4430365784`).
These UI strings are embedded in the search-results RSC stream.

In [5]:
HEADER_TERMS = [
    "Forward Deployed Engineer",
    "appliers.ai",
    "Greater Melbourne Area",
    "6 days ago",
    "Over 100 applicants",
    "Promoted by hirer",
    "Actively reviewing applicants",
    "On-site",
    "Full-time",
    "Easy Apply",
    "uncategorizedPreferences",
    "renderPayload",
]

for term in HEADER_TERMS:
    hits = [chunk_id for chunk_id, data in chunks.items() if term in data]
    print(f"{term!r}: {hits[:8]}{' ...' if len(hits) > 8 else ''}")

'Forward Deployed Engineer': ['0', '28', '29', '30', '53', 'cc', 'd2', 'd9'] ...
'appliers.ai': ['0', '28', '29', '30', '53', 'cc', 'ac', 'd2'] ...
'Greater Melbourne Area': ['28', '29', '30', '53']
'6 days ago': ['28', '29']
'Over 100 applicants': ['28', '29']
'Promoted by hirer': ['28', '29']
'Actively reviewing applicants': ['28', '29']
'On-site': ['30', '3e', '3f', '53', '59', '5a']
'Full-time': ['3e', '3f', '59', '5a']
'Easy Apply': ['cc', 'd2', 'd9', 'dd']
'uncategorizedPreferences': ['3e', '3f', '59', '5a']
'renderPayload': ['36', '54']


## topCard — metadata line, promoted badge, application status

Look for `observabilityIdentifier` = `...topcard.topCard`.

In capture 3521 the main metadata line and status badges live in chunk **`28`** (and a duplicate tree in **`29`**).

In [6]:
def iter_text_props(obj):
    if isinstance(obj, dict):
        text_props = obj.get("textProps")
        if isinstance(text_props, dict):
            text = rsc.render_text(text_props.get("children", [])).strip()
            if text:
                yield text
        for value in obj.values():
            yield from iter_text_props(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from iter_text_props(value)


def texts_for_observability(response_body: str, identifier_substring: str) -> list[tuple[str, str]]:
    out = []
    for chunk_id, data in rsc.parse_stream(response_body).items():
        if identifier_substring not in data:
            continue
        parsed = json.loads(data)
        for text in iter_text_props(parsed):
            out.append((chunk_id, text))
    return out


topcard_texts = texts_for_observability(response_body, "topcard.topCard")
for chunk_id, text in topcard_texts:
    print(f"chunk {chunk_id}: {text!r}")

chunk 28: 'Greater Melbourne Area ·6 days ago ·Over 100 applicants'
chunk 28: 'Promoted by hirer · \n## Actively reviewing applicants'
chunk 28: 'Stand out to the hiring team through building data annotation skills'
chunk 29: 'Greater Melbourne Area ·6 days ago ·Over 100 applicants'
chunk 29: 'Promoted by hirer · \n## Actively reviewing applicants'
chunk 29: 'Unable to load your job match data'
chunk 29: 'Unable to load your job match data'


In [7]:
metadata_line = next(text for cid, text in topcard_texts if "·" in text and "applicants" in text)
print("raw metadata line:", metadata_line)

location_label, listed_at_label, applicant_count_label = [
    part.strip() for part in metadata_line.split("·")
]
print("location_label:", location_label)
print("listed_at_label:", listed_at_label)
print("applicant_count_label:", applicant_count_label)

status_line = next(text for cid, text in topcard_texts if "Promoted" in text)
print("status line:", status_line)

promoted_label = "Promoted by hirer" if "Promoted by hirer" in status_line else None
application_status_label = "Actively reviewing applicants" if "Actively reviewing applicants" in status_line else None
print("promoted_label:", promoted_label)
print("application_status_label:", application_status_label)

raw metadata line: Greater Melbourne Area ·6 days ago ·Over 100 applicants
location_label: Greater Melbourne Area
listed_at_label: 6 days ago
applicant_count_label: Over 100 applicants
status line: Promoted by hirer · 
## Actively reviewing applicants
promoted_label: Promoted by hirer
application_status_label: Actively reviewing applicants


## stickyTopCard — title and company line

Compact header used when scrolling. Chunks **`30`** and **`53`** in capture 3521.

In [8]:
sticky_texts = texts_for_observability(response_body, "stickyTopCard")
for chunk_id, text in sticky_texts:
    print(f"chunk {chunk_id}: {text!r}")

title = next(text for _, text in sticky_texts if text == "Forward Deployed Engineer")
company_line = next(text for _, text in sticky_texts if "appliers.ai" in text)
print("title:", title)
print("company_line:", company_line)

company_name = company_line.split("•", 1)[0].strip()
print("company_name:", company_name)

chunk 30: 'Forward Deployed Engineer'
chunk 30: 'appliers.ai • Greater Melbourne Area (On-site)'
chunk 53: 'Forward Deployed Engineer'
chunk 53: 'appliers.ai • Greater Melbourne Area (On-site)'
title: Forward Deployed Engineer
company_line: appliers.ai • Greater Melbourne Area (On-site)
company_name: appliers.ai


## title in full topCard

The title also appears as a bold text node inside the full topCard tree (chunks `28`/`29`), not only in the sticky variant.

In [9]:
raw28 = rsc.get_chunk(response_body, "28")
idx = raw28.index("Forward Deployed Engineer")
print(raw28[max(0, idx - 120): idx + len("Forward Deployed Engineer") + 40])

k":false,"openBehaviorConfig":"$undefined"}}}}]},"viewTrackingSpecs":"$undefined","linkStyle":"$undefined","children":["Forward Deployed Engineer"]}],["$","$1","text-attr-1",{"children"


## preference pills — workplace type and employment type

Rendered as small secondary buttons (`On-site`, `Full-time`).
Structured source: `uncategorizedPreferences` inside a navigate payload (chunk **`3e`** in capture 3521).

In [10]:
pill_texts = []
for chunk_id, data in chunks.items():
    for match in re.finditer(r'"text":\["([^"]+)"\]', data):
        label = match.group(1)
        if label in {"On-site", "Full-time", "Hybrid", "Contract", "Remote"}:
            pill_texts.append((chunk_id, label))

print("pill button texts:", pill_texts[:6])

for chunk_id, data in chunks.items():
    if "uncategorizedPreferences" not in data:
        continue
    idx = data.index("uncategorizedPreferences")
    print(f"chunk {chunk_id} structured prefs:")
    print(data[idx : idx + 180])
    break

pill button texts: [('3e', 'On-site'), ('3f', 'Full-time'), ('59', 'On-site'), ('5a', 'Full-time')]
chunk 3e structured prefs:
uncategorizedPreferences":[{"isMatch":true,"name":"On-site"},{"isMatch":true,"name":"Full-time"}],"categorizedPreferences":[],"skills":[]},"requestedStateKeys":[],"requestMetadata"


## company logo

Logo URL is in a `renderPayload` block near the company name link. Chunk **`36`** in capture 3521.

In [11]:
def logo_url_from_render_payload(obj) -> str | None:
    if isinstance(obj, dict):
        render_payload = obj.get("renderPayload")
        if isinstance(render_payload, dict):
            root_url = render_payload.get("rootUrl")
            renditions = render_payload.get("imageRenditions")
            if isinstance(root_url, str) and isinstance(renditions, list) and renditions:
                best = max(
                    renditions,
                    key=lambda r: r.get("width", 0) if isinstance(r, dict) else 0,
                )
                suffix = best.get("suffixUrl") if isinstance(best, dict) else None
                if isinstance(suffix, str):
                    return root_url + suffix
                return root_url
        for value in obj.values():
            found = logo_url_from_render_payload(value)
            if found:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = logo_url_from_render_payload(value)
            if found:
                return found
    return None


logo_chunk = next(
    chunk_id
    for chunk_id, data in chunks.items()
    if "renderPayload" in data and "appliers.ai" in data
)
logo_url = logo_url_from_render_payload(json.loads(chunks[logo_chunk]))
print("logo chunk:", logo_chunk)
print("logo_url:", logo_url)

logo chunk: 36
logo_url: https://media.licdn.com/dms/image/v2/D560BAQHK4Pmq9odq5Q/company-logo_400_400/B56Zp6.HAdHkAY-/0/1762999693732/appliers_ai_logo?e=1784160000&v=beta&t=zZwFYYqHT_VEhlfhHDw6QhDELib897blyfuoFjLpbkk


## Easy Apply

Apply button label appears as `"text":["Easy Apply"]` in several chunks (e.g. `cc`, `d2` in capture 3521).

In [12]:
easy_apply_chunks = [
    chunk_id
    for chunk_id, data in chunks.items()
    if '"text":["Easy Apply"]' in data
]
print("Easy Apply chunks:", easy_apply_chunks)
print("is_easy_apply:", bool(easy_apply_chunks))

Easy Apply chunks: ['cc', 'd2', 'd9', 'dd']
is_easy_apply: True


## proposed `JobHeaderExtract` shape

Fields observed in capture 3521 — this is the target return type for `extract_job_header()` in `job_header.py`.

In [13]:
from dataclasses import asdict, dataclass


@dataclass
class JobHeaderExtract:
    job_id: str | None = None
    title: str | None = None
    company_name: str | None = None
    company_logo_url: str | None = None
    location_label: str | None = None
    listed_at_label: str | None = None
    applicant_count_label: str | None = None
    promoted_label: str | None = None
    application_status_label: str | None = None
    workplace_type_label: str | None = None
    employment_type_label: str | None = None
    is_easy_apply: bool = False


def extract_job_header(response_body: str, *, job_id: str | None = None) -> JobHeaderExtract:
    """Draft extractor — move to adapters/linkedin/extract/job_header.py."""
    chunks = rsc.parse_stream(response_body)
    topcard_texts = texts_for_observability(response_body, "topcard.topCard")
    sticky_texts = texts_for_observability(response_body, "stickyTopCard")

    result = JobHeaderExtract(job_id=job_id)

    if sticky_texts:
        texts = [text for _, text in sticky_texts]
        if texts:
            result.title = texts[0]
        if len(texts) > 1:
            result.company_name = texts[1].split("•", 1)[0].strip()

    for _, text in topcard_texts:
        if "·" in text and "applicant" in text:
            parts = [part.strip() for part in text.split("·")]
            if len(parts) >= 3:
                result.location_label, result.listed_at_label, result.applicant_count_label = parts[:3]
        if "Promoted by hirer" in text:
            result.promoted_label = "Promoted by hirer"
        if "Actively reviewing applicants" in text:
            result.application_status_label = "Actively reviewing applicants"

    pill_labels = []
    for data in chunks.values():
        for match in re.finditer(r'"text":\["([^"]+)"\]', data):
            label = match.group(1)
            if label in {"On-site", "Remote", "Hybrid"}:
                result.workplace_type_label = label
            elif label in {"Full-time", "Part-time", "Contract", "Internship"}:
                result.employment_type_label = label
            pill_labels.append(label)

    for chunk_id, data in chunks.items():
        if "renderPayload" in data and result.company_name and result.company_name in data:
            result.company_logo_url = logo_url_from_render_payload(json.loads(data))
            break

    result.is_easy_apply = any('"text":["Easy Apply"]' in data for data in chunks.values())
    return result


header = extract_job_header(response_body, job_id=job_id)
for field, value in asdict(header).items():
    print(f"{field}: {value!r}")

job_id: '4430365784'
title: 'Forward Deployed Engineer'
company_name: 'appliers.ai'
company_logo_url: 'https://media.licdn.com/dms/image/v2/D560BAQHK4Pmq9odq5Q/company-logo_400_400/B56Zp6.HAdHkAY-/0/1762999693732/appliers_ai_logo?e=1784160000&v=beta&t=zZwFYYqHT_VEhlfhHDw6QhDELib897blyfuoFjLpbkk'
location_label: 'Greater Melbourne Area'
listed_at_label: '6 days ago'
applicant_count_label: 'Over 100 applicants'
promoted_label: 'Promoted by hirer'
application_status_label: 'Actively reviewing applicants'
workplace_type_label: 'On-site'
employment_type_label: 'Full-time'
is_easy_apply: True


## extraction notes for `job_header.py`

| Field | Source in capture 3521 |
|-------|----------------------|
| `job_id` | `currentJobId` query param on request URL (pass in, or parse from URL) |
| `title` | stickyTopCard text node, or bold text in topCard |
| `company_name` | stickyTopCard line before `•`, or company link aria-label |
| `company_logo_url` | `renderPayload.rootUrl` + best `imageRenditions[].suffixUrl` |
| `location_label` | first segment of topCard metadata line (`·`-separated) |
| `listed_at_label` | second segment of metadata line |
| `applicant_count_label` | third segment of metadata line |
| `promoted_label` | topCard text containing `Promoted by hirer` |
| `application_status_label` | topCard text containing status phrase (e.g. `Actively reviewing applicants`) |
| `workplace_type_label` | preference pill button or `uncategorizedPreferences[].name` |
| `employment_type_label` | preference pill button or `uncategorizedPreferences[].name` |
| `is_easy_apply` | any chunk with `"text":["Easy Apply"]` |

This response is **not** on the component path (`/rsc-action/actions/component`). Load by capture id or filter `request_url` path to `/flagship-web/jobs/search-results/`.